<a href="https://colab.research.google.com/github/Rahul9994/ML_Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import os
REPO_URL = "https://github.com/Rahul9994/ML_Flyrank"
REPO_DIR = "ML_Flyrank"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

Cloning into 'ML_Flyrank'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 90 (delta 12), reused 76 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 1.84 MiB | 13.45 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Working dir: /content/ML_Flyrank/ML_Flyrank/ML_Flyrank


In [18]:
!pip install duckdb --quiet
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# sanity check: total rows in the whole daily fact table
con.sql(f"SELECT COUNT(*) FROM read_parquet('{base}/fact_content_daily_performance/**/*.parquet')").show()

con.sql(f"""
    SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessio

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [19]:
'''One row = one client-content-day: a specific piece of content, for a specific client,
on a specific report_date. Table: fact_content_daily_performance, filtered to
month=2026-03 (a mid-panel month, not the sealed June-2026 _sample month). Target/proxy
for my lane (Ranking Signal Analysis): gsc_ctr for that row. I deliberately exclude rows
where gsc_data_available is not TRUE, since those rows have no real observed
Search Console performance to learn from.'''

'One row = one client-content-day: a specific piece of content, for a specific client, \non a specific report_date. Table: fact_content_daily_performance, filtered to \nmonth=2026-03 (a mid-panel month, not the sealed June-2026 _sample month). Target/proxy \nfor my lane (Ranking Signal Analysis): gsc_ctr for that row. I deliberately exclude rows \nwhere gsc_data_available is not TRUE, since those rows have no real observed \nSearch Console performance to learn from.'

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [20]:
'''Feature: gsc_avg_position, gsc_impressions — knowable at the decision moment, observed
independently of the outcome I'm predicting.
Label: gsc_ctr — the outcome being explained.
Context (not modeled directly): client_hash_id, content_hash_id, report_date —
identifiers, not signals.
Excluded: rows where gsc_data_available IS NOT TRUE, and rows where client_has_gsc is
false — excluded because there's no real GSC signal to trust for these rows.'''

"Feature: gsc_avg_position, gsc_impressions — knowable at the decision moment, observed \nindependently of the outcome I'm predicting.\nLabel: gsc_ctr — the outcome being explained.\nContext (not modeled directly): client_hash_id, content_hash_id, report_date — \nidentifiers, not signals.\nExcluded: rows where gsc_data_available IS NOT TRUE, and rows where client_has_gsc is \nfalse — excluded because there's no real GSC signal to trust for these rows."

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# Query A — grain check
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT client_hash_id || content_hash_id || report_date) AS distinct_rows
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()
# Query B — row count + date span
con.sql(f"""
    SELECT COUNT(*) AS rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

# Query C — availability, IS TRUE
con.sql(f"""
    SELECT COUNT(*) AS available_rows
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").show()
# Five-feature frame ( compute ctr )
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_avg_position, gsc_impressions, gsc_clicks,
           CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    LIMIT 1000
""").df()
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┐
│ total_rows │ distinct_rows │
│   int64    │     int64     │
├────────────┼───────────────┤
│    9841378 │       9841378 │
└────────────┴───────────────┘

┌─────────┬────────────┬────────────┐
│  rows   │ start_date │  end_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘



,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,3.350000,20,0,0.000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0.000000,1,0,0.000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,4.928000,125,1,0.008
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,4.000000,7,0,0.000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,2.272727,11,0,0.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice is an unbalanced panel — client_has_gsc/client_has_ga4 show some clients
lack one data source entirely, and per dim_clients.gsc_data_start/ga4_data_start,
history depth differs by client. A single month like 2026-03 can't represent every
client equally, and this data only shows observed association, never causation

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.